# BTL02 T0304 — RQ2: pretrained vs scratch và input size

Notebook chạy tuần tự ma trận 9 cấu hình YAML trên Colab GPU. Chọn runtime T4/A100 (tối thiểu GPU CUDA), thêm `KAGGLE_API_TOKEN` vào Colab Secrets nếu FER2013 chưa có trên Drive. Checkpoint, MLflow file store, CSV 24 cột và biểu đồ được lưu trên Drive. Không ghi token vào notebook/repository. Các metrics chỉ là kết quả thực nghiệm sau khi cell hoàn tất; notebook không chứa số liệu mẫu giả.


In [ ]:
import os, platform, shutil, subprocess, sys, threading, time
from pathlib import Path

DRIVE_WORK_DIR = '/content/drive/MyDrive/BTL02/T0304_RQ2'
DRIVE_MLRUNS_DIR = f'{DRIVE_WORK_DIR}/mlruns'
DRIVE_CKPT_DIR = f'{DRIVE_WORK_DIR}/checkpoints'
DRIVE_CSV_PATH = f'{DRIVE_WORK_DIR}/t0304_rq2_transfer_vs_scratch_24col.csv'
LOCAL_MLRUNS_DIR = '/content/mlruns_t0304'
LOCAL_CKPT_DIR = '/content/checkpoints_t0304'
REPO_DIR = '/content/Deep_Learning_Group'
BRANCH = 'exp/BTL02_T0304_rq2_transfer_vs_scratch'
REPO_URL = 'https://github.com/Duy-DS/Deep_Learning_Group.git'

print(f'Python: {platform.python_version()}')
try:
    import torch
except ImportError:
    torch = None
    print('C?NH B?O: Ch?a c? PyTorch; c?i dependency ? Cell 4 r?i ch?y l?i notebook.')
if torch is not None:
    print(f'PyTorch: {torch.__version__}')
    if not torch.cuda.is_available():
        print('C?NH B?O: torch.cuda.is_available() == False ? ch?n Runtime > Change runtime type > GPU.')
        raise RuntimeError('RQ2 c?n GPU CUDA (T4/A100/L4); d?ng ?? tr?nh hu?n luy?n CPU r?t l?u.')
    for gpu_index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(gpu_index)
        vram_gib = props.total_memory / (1024 ** 3)
        print(f'GPU {gpu_index}: {props.name} | VRAM {vram_gib:.1f} GiB')
        if not any(name in props.name.upper() for name in ('T4', 'A100', 'L4')):
            print('C?NH B?O: GPU kh?ng thu?c T4/A100/L4 d? ki?n.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

for directory in (DRIVE_WORK_DIR, DRIVE_MLRUNS_DIR, DRIVE_CKPT_DIR, LOCAL_CKPT_DIR):
    Path(directory).mkdir(parents=True, exist_ok=True)
Path(LOCAL_MLRUNS_DIR, '.trash').mkdir(parents=True, exist_ok=True)

def sync_tree_new_files(source, destination, stable_age_seconds=0):
    source_path, destination_path = Path(source), Path(destination)
    destination_path.mkdir(parents=True, exist_ok=True)
    if not source_path.exists():
        return
    for current_root, directories, filenames in os.walk(source_path):
        current = Path(current_root)
        target_root = destination_path / current.relative_to(source_path)
        target_root.mkdir(parents=True, exist_ok=True)
        directories[:] = [name for name in directories if name != '.trash']
        for filename in filenames:
            source_file = current / filename
            if source_file.is_symlink() or filename.endswith(('.tmp', '.part', '.sync-tmp')):
                continue
            source_stat = source_file.stat()
            if stable_age_seconds and time.time() - source_stat.st_mtime < stable_age_seconds:
                continue
            target_file = target_root / filename
            if target_file.exists() and source_stat.st_size == target_file.stat().st_size and source_stat.st_mtime_ns <= target_file.stat().st_mtime_ns:
                continue
            temporary_file = target_file.with_name(target_file.name + '.sync-tmp')
            shutil.copy2(source_file, temporary_file)
            copied_stat = source_file.stat()
            temporary_stat = temporary_file.stat()
            if copied_stat.st_size != source_stat.st_size or copied_stat.st_mtime_ns != source_stat.st_mtime_ns or temporary_stat.st_size != source_stat.st_size:
                temporary_file.unlink(missing_ok=True)
                continue
            os.replace(temporary_file, target_file)

# Resume: merge prior Drive state into the fresh Colab SSD without deleting local files.
sync_tree_new_files(DRIVE_MLRUNS_DIR, LOCAL_MLRUNS_DIR)
sync_tree_new_files(DRIVE_CKPT_DIR, LOCAL_CKPT_DIR)
Path(LOCAL_MLRUNS_DIR, '.trash').mkdir(parents=True, exist_ok=True)
os.environ['MLFLOW_TRACKING_URI'] = Path(LOCAL_MLRUNS_DIR).as_uri()
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
print('MLflow local-first URI:', os.environ['MLFLOW_TRACKING_URI'])
print('Resume sync completed from Drive.')


In [ ]:
from google.colab import userdata
github_token = None
try:
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    print('Kh?ng ??c ???c Colab Secret GITHUB_TOKEN. N?u repository private, th?m secret trong Colab Secrets v? b?t Notebook access; repository public c? th? clone kh?ng token.')

git_env = os.environ.copy()
askpass_path = None
if github_token:
    askpass = '#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$COLAB_GITHUB_TOKEN" ;; esac\n'
    import tempfile
    descriptor, askpass_path = tempfile.mkstemp(prefix='colab-git-askpass-')
    with os.fdopen(descriptor, 'w', encoding='utf-8') as stream:
        stream.write(askpass)
    os.chmod(askpass_path, 0o700)
    git_env.update({'GIT_ASKPASS': askpass_path, 'GIT_TERMINAL_PROMPT': '0', 'COLAB_GITHUB_TOKEN': github_token})
try:
    if Path(REPO_DIR).exists():
        subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True, env=git_env)
        subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True, env=git_env)
    else:
        subprocess.run(['git', 'clone', '-b', BRANCH, '--single-branch', REPO_URL, REPO_DIR], check=True, env=git_env)
finally:
    if askpass_path and os.path.exists(askpass_path):
        os.remove(askpass_path)
    git_env.pop('COLAB_GITHUB_TOKEN', None)
    github_token = None

PROJECT_DIR = Path(REPO_DIR) / 'BTL02'
os.chdir(PROJECT_DIR)
print('Repository branch:', subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip())

outputs_dir = Path('outputs')
outputs_dir.mkdir(parents=True, exist_ok=True)
checkpoint_link = outputs_dir / 'checkpoints'
if checkpoint_link.is_symlink():
    if checkpoint_link.resolve() != Path(LOCAL_CKPT_DIR).resolve():
        raise RuntimeError(f'Checkpoint symlink ?ang tr? sai v? tr?: {checkpoint_link.resolve()}')
elif checkpoint_link.exists():
    if checkpoint_link.is_dir():
        existing_items = list(checkpoint_link.iterdir())
        if all(item.name == '.gitkeep' and item.is_file() for item in existing_items):
            for item in existing_items:
                item.unlink()
            checkpoint_link.rmdir()
        else:
            raise RuntimeError('outputs/checkpoints ?? c? d? li?u; kh?ng ghi ??. H?y sao l?u/di chuy?n th? c?ng tr??c khi ch?y.')
    else:
        raise RuntimeError('outputs/checkpoints t?n t?i nh?ng kh?ng ph?i symlink; d?ng ?? b?o to?n d? li?u.')
if not checkpoint_link.exists():
    checkpoint_link.symlink_to(Path(LOCAL_CKPT_DIR), target_is_directory=True)
print('Checkpoint output ->', checkpoint_link.resolve())


In [ ]:
requirements = Path('requirements.txt').read_text(encoding='utf-8').splitlines()
filtered_requirements = [
    line for line in requirements
    if line.strip() and not line.lstrip().startswith('#')
    and line.split('=', 1)[0].strip().lower() not in {'torch', 'torchvision'}
]
if sys.version_info >= (3, 13):
    filtered_requirements = [
        'scikit-learn==1.6.1' if line.strip().lower().startswith('scikit-learn==') else line
        for line in filtered_requirements
    ]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *filtered_requirements], check=True)

import torch, torchvision
print('Ready PyTorch/Torchvision:', torch.__version__, torchvision.__version__)
if not torch.cuda.is_available():
    print('CẢNH BÁO: không có GPU CUDA; tiếp tục sẽ rất chậm.')
else:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f'GPU: {gpu_name} | VRAM: {gpu_memory_gib:.1f} GiB')
    if gpu_memory_gib < 12:
        raise RuntimeError(f'VRAM {gpu_memory_gib:.1f} GiB dưới ngưỡng an toàn 12 GiB cho ma trận này.')

raw_csv = Path('data/raw/fer2013.csv')
drive_csv = Path(DRIVE_WORK_DIR) / 'fer2013.csv'
raw_csv.parent.mkdir(parents=True, exist_ok=True)
if not raw_csv.is_file() and drive_csv.is_file():
    shutil.copy2(drive_csv, raw_csv)
if not raw_csv.is_file():
    try:
        kaggle_token = userdata.get('KAGGLE_API_TOKEN')
    except Exception:
        kaggle_token = None
    if not kaggle_token:
        raise RuntimeError('Thiếu Colab Secret KAGGLE_API_TOKEN. Thêm secret và bật Notebook access; cần quyền truy cập cuộc thi FER2013 trên Kaggle.')
    os.environ['KAGGLE_API_TOKEN'] = kaggle_token
    del kaggle_token
    try:
        subprocess.run([sys.executable, '-m', 'src.data.downloader'], check=True, env=os.environ.copy())
    finally:
        os.environ.pop('KAGGLE_API_TOKEN', None)
from src.data.downloader import FER2013Downloader
split_counts = FER2013Downloader.validate_csv(raw_csv)
assert split_counts == {'Training': 28709, 'PublicTest': 3589, 'PrivateTest': 3589}, split_counts
if not drive_csv.is_file():
    shutil.copy2(raw_csv, drive_csv)
print('FER2013 verified:', split_counts, '| total:', sum(split_counts.values()))


In [ ]:
import numpy as np
import pandas as pd
import yaml

CONFIG_FILES = [
    'exp04_vgg16_finetune.yaml', 'exp05_resnet50_finetune.yaml', 'exp06_efficientnet_b0_finetune.yaml',
    'exp04b_vgg16_scratch_128.yaml', 'exp05b_resnet50_scratch_128.yaml', 'exp06b_efficientnet_b0_scratch_128.yaml',
    'exp04c_vgg16_pretrained_48.yaml', 'exp05c_resnet50_pretrained_48.yaml', 'exp06c_efficientnet_b0_pretrained_48.yaml',
]

def sanitize_to_python_scalars(record: dict) -> dict:
    def sanitize(value):
        if isinstance(value, np.generic):
            return sanitize(value.item())
        if isinstance(value, np.ndarray):
            return sanitize(value.item()) if value.ndim == 0 else str(sanitize(value.tolist()))
        if isinstance(value, dict):
            return str({str(key): sanitize(item) for key, item in value.items()})
        if isinstance(value, (list, tuple)):
            return str([sanitize(item) for item in value])
        if value is None or type(value) in (bool, int, float, str):
            return value
        return str(value)
    return {str(key): sanitize(value) for key, value in record.items()}

raw_configs = []
for filename in CONFIG_FILES:
    config_path = Path('experiments/sota') / filename
    config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
    config['config_path'] = str(config_path)
    config['run_name'] = f"{config['exp_id']}_{config['architecture']}_{'pretrained' if config['pretrained'] else 'scratch'}_{config['image_size']}_{config['optimizer'].lower()}"
    raw_configs.append(config)
assert len(raw_configs) == 9
manifest = pd.DataFrame([sanitize_to_python_scalars(record) for record in raw_configs])
manifest = manifest.astype({'pretrained': bool, 'image_size': int, 'seed': int, 'batch_size': int, 'epochs': int, 'lr': float})

from mlflow.tracking import MlflowClient
client = MlflowClient(tracking_uri=os.environ['MLFLOW_TRACKING_URI'])
experiment = client.get_experiment_by_name('BTL02_FER2013_PyTorch')
finished_names = set()
if experiment is not None:
    for run in client.search_runs([experiment.experiment_id], max_results=10000):
        if run.info.status == 'FINISHED' and run.data.tags.get('mlflow.runName'):
            finished_names.add(run.data.tags['mlflow.runName'])
manifest['status'] = ['DONE' if name in finished_names else 'PENDING' for name in manifest['run_name']]
display(manifest[['exp_id', 'architecture', 'pretrained', 'image_size', 'seed', 'batch_size', 'epochs', 'lr', 'run_name', 'status']])


In [ ]:
from src.train.mlflow_logger import MLflowLogger
LOCAL_RESULTS_CSV = Path('outputs/logs/t0304_rq2_transfer_vs_scratch_24col.csv')
Path(LOCAL_MLRUNS_DIR, '.trash').mkdir(parents=True, exist_ok=True)
sync_lock = threading.Lock()

def sync_and_export_results(sync_checkpoints=False, drive_sync_stability_seconds=0):
    with sync_lock:
        logger = MLflowLogger()
        raw_export = Path('outputs/logs/t0304_all_runs_export.csv')
        logger.export_summary_csv(str(raw_export))
        all_rows = pd.read_csv(raw_export, encoding='utf-8-sig', keep_default_na=False)
        assert all_rows.columns.tolist() == MLflowLogger.CSV_COLUMNS
        rq2_ids = set(manifest['exp_id'].astype(str))
        selected = all_rows[all_rows['Status'].astype(str).str.upper().eq('FINISHED')].copy()
        selected = selected[selected['Name'].astype(str).isin(set(manifest['run_name']))].copy()
        name_to_config = {row['run_name']: row for row in raw_configs}
        selected['params.exp_id'] = selected['Name'].map(lambda name: name_to_config[str(name)]['exp_id'] if str(name) in name_to_config else '')
        selected = selected[selected['params.exp_id'].astype(str).isin(rq2_ids)].copy()
        safe_rows = MLflowLogger.normalize_records(selected.to_dict(orient='records'))
        result = pd.DataFrame(safe_rows, columns=MLflowLogger.CSV_COLUMNS + ['params.exp_id'])
        local_csv_tmp = LOCAL_RESULTS_CSV.with_name(LOCAL_RESULTS_CSV.name + '.sync-tmp')
        result[MLflowLogger.CSV_COLUMNS].to_csv(local_csv_tmp, index=False, encoding='utf-8-sig')
        os.replace(local_csv_tmp, LOCAL_RESULTS_CSV)
        drive_csv = Path(DRIVE_CSV_PATH)
        drive_csv.parent.mkdir(parents=True, exist_ok=True)
        drive_csv_tmp = drive_csv.with_name(drive_csv.name + '.sync-tmp')
        shutil.copy2(LOCAL_RESULTS_CSV, drive_csv_tmp)
        os.replace(drive_csv_tmp, drive_csv)
        sync_tree_new_files(LOCAL_MLRUNS_DIR, DRIVE_MLRUNS_DIR, stable_age_seconds=drive_sync_stability_seconds)
        if sync_checkpoints:
            sync_tree_new_files(LOCAL_CKPT_DIR, DRIVE_CKPT_DIR)
        raw_export.unlink(missing_ok=True)
        print(f'Synced {len(result)} FINISHED RQ2 runs; CSV: {DRIVE_CSV_PATH}', flush=True)
        return result

print('CSV 24-column destination:', DRIVE_CSV_PATH)
print('MLflow sync every 60 seconds; checkpoints sync after each run completes.')


In [ ]:
sync_errors = []
sync_stop = threading.Event()

def periodic_drive_sync():
    while not sync_stop.wait(60):
        try:
            sync_and_export_results(sync_checkpoints=False, drive_sync_stability_seconds=5)
        except Exception as error:
            sync_errors.append(repr(error))
            print(f'[SYNC WARNING] Drive sync failed; SSD state remains available: {error!r}', flush=True)

sync_thread = threading.Thread(target=periodic_drive_sync, name='t0304-drive-sync', daemon=True)
sync_thread.start()
failed_runs = []

try:
    for config in raw_configs:
        run_name = config['run_name']
        experiment = client.get_experiment_by_name('BTL02_FER2013_PyTorch')
        is_done = False
        if experiment is not None:
            matches = client.search_runs([experiment.experiment_id], filter_string=f"tags.mlflow.runName = '{run_name}'", max_results=100)
            is_done = any(run.info.status == 'FINISHED' for run in matches)
        if is_done:
            print(f'[SKIP] ?? ho?n t?t tr??c ??: {run_name}', flush=True)
            continue
        config_path = config['config_path']
        print(f'\n===== RUN {config["exp_id"]}: {run_name} =====', flush=True)
        command = [sys.executable, '-u', '-m', 'src.train.trainer', '--config', config_path]
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ.copy()) as process:
            for output_line in process.stdout:
                print(output_line, end='', flush=True)
            return_code = process.wait()
        if return_code:
            failed_runs.append({'run_name': run_name, 'return_code': return_code})
            print(f'[FAILED] {run_name} (exit={return_code}); ??ng b? an to?n v? ti?p t?c run k? ti?p.', flush=True)
        else:
            print(f'[DONE] {run_name}', flush=True)
        try:
            sync_and_export_results(sync_checkpoints=(return_code == 0))
        except Exception as error:
            sync_errors.append(repr(error))
            print(f'[SYNC WARNING] Kh?ng ??ng b? ???c sau {run_name}; run local v?n c?n tr?n SSD: {error!r}', flush=True)
finally:
    sync_stop.set()
    sync_thread.join(timeout=10)
    try:
        sync_and_export_results(sync_checkpoints=True)
    except Exception as error:
        sync_errors.append(repr(error))
        print(f'[SYNC WARNING] Final Drive sync failed: {error!r}', flush=True)

if failed_runs:
    print('Runs c?n ch?y l?i:', failed_runs)
print('Training loop completed; FINISHED runs will be skipped on the next Run all.')


In [ ]:
import matplotlib.pyplot as plt
try:
    final_results = sync_and_export_results(sync_checkpoints=True)
except Exception as error:
    print(f'[SYNC WARNING] Hi?n th? b?n CSV g?n nh?t c? tr?n Drive: {error!r}')
    local_csv = Path('outputs/logs/t0304_rq2_transfer_vs_scratch_24col.csv')
    final_results = pd.read_csv(local_csv, encoding='utf-8-sig', keep_default_na=False) if local_csv.is_file() else pd.DataFrame(columns=MLflowLogger.CSV_COLUMNS)
if Path(DRIVE_CSV_PATH).is_file():
    final_results = pd.read_csv(DRIVE_CSV_PATH, encoding='utf-8-sig', keep_default_na=False)
display(final_results)
if not final_results.empty:
    result_with_design = manifest.merge(final_results, left_on='run_name', right_on='Name', how='left', suffixes=('_design', ''))
    metric_columns = [column for column in ('metrics.best_val_acc', 'metrics.private_test_acc', 'metrics.macro_f1') if column in result_with_design.columns]
    display(result_with_design[['exp_id', 'architecture', 'pretrained', 'image_size', 'Status', *metric_columns]])
    result_with_design['architecture_label'] = result_with_design['architecture'].replace({'EfficientNetB0': 'EfficientNet-B0'})
    figure, axes = plt.subplots(1, 3, figsize=(19, 5))
    for axis, metric in zip(axes, metric_columns):
        values = pd.to_numeric(result_with_design[metric], errors='coerce')
        axis.bar(result_with_design['architecture_label'] + '/' + result_with_design['image_size'].astype(str) + 'px', values)
        axis.set_title(metric)
        axis.tick_params(axis='x', rotation=75)
    figure.tight_layout()
    display(figure)
print('RQ2 CSV tr?n Google Drive:', DRIVE_CSV_PATH)
print('Axes: VGG16/ResNet50/EfficientNet-B0; pretrained/scratch; image_size 48/128.')
